In [0]:
SELECT
  solution_architect_user_name as solution_architect,
  --target_live_fiscal_year_quarter,
  --usecase_name,
  --stage_number,
  -- days_in_stage,
  -- days_in_validating,
  -- days_in_scoping,
  --days_in_evaluating, 
  --days_in_evaluating_all_time, 
  --days_in_confirming,
  --evaluating_date,
  --confirming_date,
  --onboarding_date,
  --live_date,

  date_format(add_months(confirming_date, 11), 'yyyy') as confirming_date_fiscal,
  date_format(dateadd(year, +1, dateadd(month, -1, confirming_date)), "'FY'yy'-Q'Q") as confirming_date_fq,
  SUM(CASE WHEN stage_number > 3 AND days_in_evaluating_all_time = 0 THEN 1 END) AS uco_count_eval_skipped,    
  SUM(CASE WHEN stage_number > 3 AND days_in_evaluating > 0 AND days_in_evaluating <= 60 THEN 1 END) as eval_lt_60_days_uco_count,
  SUM(CASE WHEN stage_number > 3 AND days_in_evaluating > 0 THEN days_in_evaluating END) AS past_eval_days_sum,
  NULLIF(COUNT(CASE WHEN stage_number > 3 AND days_in_evaluating > 0 THEN 1 END), 0) as past_eval_total_uco_count
  --COUNT(try_divide(eval_lt_60_days_uco_count, past_eval_total_uco_count)) AS pct_won_within_60_days,
  --ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY CASE WHEN stage_number > 3 AND days_in_evaluating > 0 THEN days_in_evaluating END), 0) AS median_days_in_evaluating,
  --ROUND(AVG(CASE WHEN stage_number > 3 AND days_in_evaluating > 0 THEN days_in_evaluating END), 0) AS avg_days_in_evaluating,
  --try_divide(past_eval_days_sum, past_eval_total_uco_count) as avg_days_in_evaluating_calc,

FROM main.gtm_gold.rpt_use_case_detail
WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE CONCAT('%', :ae_email, '%')
--AND sales_subregion_level_3 = 'Italy Strategic Core'
--AND deployable_account_name NOT IN ('Gruppo Hera', 'ALIA SERVIZI AMBIENTALI SPA', 'Snam Spa') --interim accounts
AND date_format(add_months(confirming_date, 11), 'yyyy') >= 2026
AND stage_number > 3 --UCOs which completed evaluation.
AND solution_architect_user_name IS NOT NULL
  
GROUP BY ALL